# Train YOLOv8n-P2 Drone V5 (Dataset Cận Cảnh ⮕ Inference Drone 3-4m)

**So với V4 (`yolov8n` cổ điển):** thêm **P2 head** — tầng feature map co ít nhất (×4, 160×160) → nhìn được **lá bệnh nhỏ vài pixel** mà P3 (80×80) bỏ sót. Nhắm đúng bài toán drone bay cao.

**P2 là gì:** backbone YOLOv8 xuất 3 feature map P3/8, P4/16, P5/32. Model này thêm nhánh **P2/4** ở tầng chi tiết nhất → cùng một lá 8px, ở P3 nằm trong 1 cell còn ở P2 nằm trong 4 cell → dễ phát hiện hơn rõ.

**Cái giá phải trả:** model nặng hơn đáng kể so với `yolov8n` (thêm nhánh ở feature map 160×160), chậm hơn trên K230 & điện thoại. Nếu FPS là ưu tiên → quay lại v4 + SAHI 320.

**⚠ Quan trọng — Vì sao không dùng pretrained `yolov8n.pt`:**
- `YOLO("yolov8n.pt").load("yolov8n-p2.yaml")` chỉ dùng được khi **cùng kiến trúc head**. P2 có 4 nhánh Detect, mặc định chỉ 3 → load sẽ lỗi.
- Remote weight `yolov8n-p2.pt` **mới có nếu ultralytics ≥ 8.3** (P2 8.3.0 mới phát hành head P2). Nếu dùng ultralytics ≥ 8.3: dùng `YOLO("yolov8n-p2.pt")`.
- Như nhận độ an toàn + nc=39, notebook này dùng `YOLO("yolov8n-p2.yaml")` train **từ đầu** (COCO-pretrained backbone không bắt buộc; `pretrained=False` mặc định).


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow sahi opencv-python

from ultralytics import YOLO
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Cần >= 8.3 để nhận dạng yolov8n-p2.pt (nếu đã có remote weight); cảnh báo nếu thấp hơn
major, minor = map(int, ultralytics.__version__.split(".")[:2])
if not (major > 8 or (major == 8 and minor >= 3)):
    print("⚠ WARNING: ultralytics < 8.3 — chuẩn bị dùng yolov8n-p2.yaml train từ đầu (cách an toàn).")

## 1. Tải Dataset Cận Cảnh từ Roboflow

In [ ]:
def _get_roboflow_key():
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key: return key
    except Exception: pass
    if os.environ.get("ROBOFLOW_API_KEY"): return os.environ["ROBOFLOW_API_KEY"]
    try:
        for line in open(".env"):
            if line.startswith("ROBOFLOW_API_KEY") and "=" in line:
                return line.split("=", 1)[1].strip()
    except FileNotFoundError: pass
    raise ValueError("Thiếu ROBOFLOW_API_KEY. Thêm secret tên ROBOFLOW_API_KEY trên Kaggle!")

ROBOFLOW_API_KEY = _get_roboflow_key()
print("Đã lấy ROBOFLOW_API_KEY")

from roboflow import Roboflow
WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset_info = project.version(PROJECT_VERSION).download("yolov8")
print("Dataset cận cảnh đã tải về /kaggle/working/")

In [ ]:
import os, glob, yaml

candidates = glob.glob("/kaggle/working/*/data.yaml")
DATASET_PATH = os.path.dirname(candidates[0]) if candidates else "/kaggle/working/citrus-disease-detection-1"
TRAIN_DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")
print("DATASET_PATH =", DATASET_PATH)

with open(TRAIN_DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print("Số class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load Model N-P2

Dùng `yolov8n-p2.yaml` (9.6M params, ~2.4x yolov8n 3.2M) — có sẵn trong ultralytics ≥ 8.3.

Nếu muốn khởi tạo từ COCO weights đã train sẵn (thay vì train thật từ đầu):
1. Chạy cell này với `YOLO("yolov8n-p2.pt")` (nếu ultralytics ≥ 8.3 có remote weight).
2. Hoặc download `yolov8n.pt`, đổi key `model.yaml` bên trong thành yaml P2, rồi `load()`.

Cách an toàn nhất (nhận 39 class, không phụ thuộc remote weight): train từ đầu bằng yaml P2.

In [ ]:
MODEL_YAML = "yolov8n-p2.yaml"
MODEL_NAME = "yolov8n-p2"

# Tạo model từ yaml — nc được đẩy từ data yaml lúc train
model = YOLO(MODEL_YAML)
print(f"Đã load kiến trúc {MODEL_YAML} (P2/4 + P3/8 + P4/16 + P5/32) — chưa load weights, train từ đầu.")

In [ ]:
# ===== CẤU HÌNH TRAIN P2 CHO DATASET CẬN CẢNH ROBOFLOW =====
EPOCHS   = 150
IMGSZ    = 640        # Chuẩn resolution cho K230 ONNX export
BATCH    = 16         # Batch nhỏ hơn v4 vì V-P2 nặng gấp ~2.4x (dùng RAM/VRAM giữ mức)
PATIENCE = 20

OUT_DIR = "/kaggle/working/drone_yolo_v5_p2_close_up"
os.makedirs(OUT_DIR, exist_ok=True)

# Auto-backup best.pt sau mỗi epoch
import shutil
from ultralytics.utils import callbacks

def _backup(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))
        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}/best_checkpoint.pt", flush=True)
    except Exception as e:
        pass

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

print(f"Train {MODEL_NAME}: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

# Augmentations giúp model cận cảnh thích nghi với ảnh lá nhỏ từ xa (giống v4)
train_args = dict(
    data=TRAIN_DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    time=8,              # Tối đa 8 giờ session Kaggle
    cache=True,
    workers=2,
    # Augmentations biến đổi scale & góc chụp
    scale=0.8,           # Thu nhỏ/phóng to ngẫu nhiên từ 20% đến 180% kích thước lá
    fliplr=0.5,          # Lật ngang
    mosaic=1.0,          # Mosaic 4 ảnh cận cảnh ghép lại (tạo góc nhìn nhiều lá)
    mixup=0.15,          # Trộn ảnh tạo nhiễu ánh sáng
    copy_paste=0.2,      # Trộn vết bệnh
    cos_lr=True,         # Cosine LR decay
    project="/kaggle/working/runs",
    name="drone_yolov8n_p2_closeup",
)

results = model.train(**train_args)

## 3. Đánh giá & Export Model (pt + onnx)

File **`best.onnx`** sẽ được export để phục vụ chuyển đổi sang **`best.kmodel`** chạy trên chip Kendryte K230 của Drone.

**⚠ Lưu ý export:** model sau train có 4 output gồm **P2 ở index 0** (cùng format `(nc+4, 8400)` attribute-major như v4). Khi convert sang NCNN/kmodel và viết code detect, hardcode như sau:
- Nếu model **toàn đầu** (pretrained toàn ảnh input bất kỳ): bình thường NCNN event `out0 = P5/32`, `out1 = P4/16`, `out2 = P3/8`, **`out3 = P2/4`** → **nhớ đọc out3 riêng**, đừng để shape mismatch gây out-of-bounds.
- Nếu export với `nms=True`: nhập lại logic khớp stride.

→ **Code detect hiện tại của app Android hoặc `model_ncnn.py` cần cập nhật nếu dùng model P2**; không tương thích được ngay với audio layout bỏ đi (4 output thay vì 3).

In [ ]:
metrics = model.val()
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
# Export ONNX dành cho chip K230 Drone
RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n_p2_closeup/weights"
best_path   = os.path.join(RESULTS_DIR, "best.pt")

if os.path.exists(best_path):
    best_model = YOLO(best_path)
    best_model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)
    shutil.copy(best_path, os.path.join(OUT_DIR, "best.pt"))
    onnx_src = os.path.join(RESULTS_DIR, "best.onnx")
    if os.path.exists(onnx_src):
        shutil.copy(onnx_src, os.path.join(OUT_DIR, "best.onnx"))
    print(f"Đã copy best.pt và best.onnx vào {OUT_DIR}")
else:
    print(f"Không tìm thấy {best_path}")

print("\n>>> TẢI KẾT QUẢ: Panel bên phải tab 'Output' -> biểu tượng Download all.")

## Ghi chú sau khi train xong

- **So sánh với v4:** mAP50 của P2 thường thấp hơn yolo8n 'n' trên dataset CN (vì head to hơn, học lâu hơn) nhưng recall trên vật nhỏ tốt hơn. Xét `metrics.box.mr` (recall) và thực địa trên drone.
- **Nếu muốn dùng trên Android app hiện tại:** bắt buộc cập nhật `yolov8_det.cpp` để đọc output index 0 (P2) và các stride khác — hoặc export thêm bản strip-P2 nếu muốn giữ nguyên code cũ.
- **K230:** convert kmodel từ ONNX này tương tự v1/v2. Với v5 (P2) thì model nặng hơn — kiểm tra RAM NPU trước khi deploy.